In [6]:
# ==========================================
# 1. Mount Google Drive
# ==========================================
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
# ==========================================
# 2. Import Library & Setup Direktori
# ==========================================
import pandas as pd
import glob
import os

# Base path direktori penelitian Anda
base_path = "/content/drive/My Drive/skripsi/dataset/mbg_2025/"

# Direktori pecahan file (batch)
seed_dir = os.path.join(base_path, "labeling_roberta/seed_batches_label_studio/*.csv")
refill_dir = os.path.join(base_path, "labeling_roberta/refill_batches_label_studio/*.csv")

# Path file Master
master_path = os.path.join(base_path, "merged_data/merged_mbg_full_with_text_duplication.csv")

# Path output file
output_dir = os.path.join(base_path, "processed/")
output_path = os.path.join(output_dir, "mbg_sampled_research.csv")

# Membuat folder output jika belum ada
os.makedirs(output_dir, exist_ok=True)

In [8]:
# ==========================================
# 3. Ekstraksi dan Penggabungan ID
# ==========================================
print("Mulai mengumpulkan ID dari file pecahan...")

# Menggunakan set() untuk memastikan ID unik (tidak ada duplikat)
unique_ids = set()

# Fungsi bantuan untuk membaca ID dari sekumpulan file
def extract_ids_from_files(file_pattern, source_name):
    files = glob.glob(file_pattern)
    print(f"Ditemukan {len(files)} file di direktori {source_name}.")

    count = 0
    for file in files:
        # Hanya membaca kolom 'id_string' untuk efisiensi memori, dipaksa menjadi string
        try:
            df_temp = pd.read_csv(file, usecols=['id_string'], dtype={'id_string': str})
            # Membersihkan spasi (jika ada) dan memasukkan ke dalam set
            batch_ids = df_temp['id_string'].str.strip().dropna().tolist()
            unique_ids.update(batch_ids)
            count += len(batch_ids)
        except Exception as e:
            print(f"⚠️ Gagal membaca {os.path.basename(file)}: {e}")

    print(f"Total ID yang diekstrak dari {source_name} (termasuk duplikat antar file): {count}")

# Eksekusi ekstraksi
extract_ids_from_files(seed_dir, "Seed Batches")
extract_ids_from_files(refill_dir, "Refill Batches")

print(f"\n✅ Total ID Unik yang berhasil dikumpulkan: {len(unique_ids)}")

Mulai mengumpulkan ID dari file pecahan...
Ditemukan 12 file di direktori Seed Batches.
Total ID yang diekstrak dari Seed Batches (termasuk duplikat antar file): 2400
Ditemukan 24 file di direktori Refill Batches.
Total ID yang diekstrak dari Refill Batches (termasuk duplikat antar file): 4440

✅ Total ID Unik yang berhasil dikumpulkan: 6809


In [9]:
# ==========================================
# 4. Load Master File & Filtering Data
# ==========================================
print("Sedang memuat data master... (Ini mungkin memakan waktu jika file besar)")

# Membaca data master
master_df = pd.read_csv(
    master_path,
    dtype={'id_str': str},
    encoding_errors='ignore',
    low_memory=False
)

print(f"Data master berhasil dimuat. Total baris: {len(master_df)}")

# Membersihkan spasi (whitespace) pada kolom id_str agar pencocokan akurat
master_df['id_str'] = master_df['id_str'].astype(str).str.strip()

# ==========================================
# Pengecekan ID Cocok dan Tidak Cocok
# ==========================================
ids_in_master = set(master_df['id_str'].tolist())

# Menggunakan set operations untuk memisahkan ID yang ada dan tidak ada
found_ids = unique_ids.intersection(ids_in_master)
missing_ids = unique_ids - ids_in_master

print(f"\n🔍 Hasil Pencocokan:")
print(f"✅ Ditemukan {len(found_ids)} ID yang cocok.")

if len(missing_ids) > 0:
    print(f"⚠️ Melewati {len(missing_ids)} ID karena tidak ditemukan di file master.")
else:
    print("✨ Sempurna! Semua ID dari batch ditemukan di file master.")

# Melakukan filtering: Hanya buat row dari ID yang benar-benar ada (found_ids)
sampled_df = master_df[master_df['id_str'].isin(found_ids)].copy()

Sedang memuat data master... (Ini mungkin memakan waktu jika file besar)
Data master berhasil dimuat. Total baris: 158712

🔍 Hasil Pencocokan:
✅ Ditemukan 6809 ID yang cocok.
✨ Sempurna! Semua ID dari batch ditemukan di file master.


In [10]:
# ==========================================
# 5. Menyimpan Hasil (mbg_sampled.csv)
# ==========================================
print(f"Menyimpan data sampel ke:\n{output_path}")

# Menyimpan dataframe hasil filter tanpa kolom index
sampled_df.to_csv(output_path, index=False)

print("\n🎉 Proses Pembuatan Sampled Data Selesai!")
print("================== Summary ==================")
print(f"1. Total ID unik dari batch (Target) : {len(unique_ids)}")
print(f"2. Total ID yang COCOK (Berhasil)    : {len(sampled_df)}")
print(f"3. Total ID yang DILEWATI (Missing)  : {len(missing_ids)}")
print(f"---------------------------------------------")
print(f"4. Total baris di data master        : {len(master_df)}")
print(f"5. Total kolom di output data        : {len(sampled_df.columns)} (Tetap utuh dari master)")
print("=============================================")

# Tampilkan 3 sampel teratas untuk memastikan data aman
sampled_df.head(3)

Menyimpan data sampel ke:
/content/drive/My Drive/skripsi/dataset/mbg_2025/processed/mbg_sampled_research.csv

🎉 Proses Pembuatan Sampled Data Selesai!
================== Summary ==================
1. Total ID unik dari batch (Target) : 6809
2. Total ID yang COCOK (Berhasil)    : 6809
3. Total ID yang DILEWATI (Missing)  : 0
---------------------------------------------
4. Total baris di data master        : 158712
5. Total kolom di output data        : 16 (Tetap utuh dari master)


,conversation_id_str,created_at,favorite_count,full_text,id_str,image_url,in_reply_to_screen_name,lang,location,quote_count,reply_count,retweet_count,tweet_url,user_id_str,username,source_keyword
4,1877134528675717178,Wed Jan 08 23:25:21 +0000 2025,0,Salut buat kemajuan MBG berkat keterlibatan pe...,1877134528675717178,https://pbs.twimg.com/media/Ggzr5xAaMAEQM_Z.jpg,NaN,in,NaN,0,0,0,https://x.com/undefined/status/187713452867571...,1698915447553597440,;;;;;;;,mbg
7,1877134662981615843,Wed Jan 08 23:25:53 +0000 2025,0,Asik bener kehadiran UMKM dalam bantu MBG kita...,1877134662981615843,https://pbs.twimg.com/media/GgzsBk4aMAIG9BM.jpg,NaN,in,NaN,0,0,0,https://x.com/undefined/status/187713466298161...,1698936320822013952,;;;;;;;,mbg
10,1877135317049725312,Wed Jan 08 23:28:29 +0000 2025,0,UMKM kita emang top bantu suksesnya program MB...,1877135317049725312,https://pbs.twimg.com/media/Ggzsnn7aMAMe2Hc.jpg,NaN,in,NaN,0,0,0,https://x.com/undefined/status/187713531704972...,1701441875876675584,;;;;;;;,mbg
